# 03 · GAE、bootstrap 与 Critic targets

## 学习目标

分开观察 TD bootstrap mask 和 GAE recursion mask，手算再调用源码验证。

默认运行参考实现。练习 Cell 需要主动启用，修改 `.py` 后重新运行。

## 环境准备

先按 [使用指南](README.md) 安装 `.[dev,notebook]`，选择项目 Kernel。

In [1]:
from pathlib import Path
import os
import sys
import torch
import numpy as np

# 从仓库根目录或 notebooks/ 打开都可；不要依赖某台机器的绝对路径。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'pyproject.toml').exists()
                     and (p / 'src/ppo_pendulum').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('请从项目目录启动 JupyterLab')
os.chdir(PROJECT_ROOT)
%load_ext autoreload
%autoreload 2
torch.manual_seed(7)
np.random.seed(7)
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT.name)

Python: /Users/matt/Documents/Codex/2026-11-15/referenced-chatgpt-conversation-this-is-an-2/work/.venv/bin/python
Project: ppo-inverted-pendulum


## 1. 两个 mask

    $$\delta_t=r_t+\gamma(1-\mathrm{terminated}_t)V(s_{t+1})-V(s_t)$$
    $$\hat A_t=\delta_t+\gamma\lambda(1-(\mathrm{terminated}_t\lor\mathrm{truncated}_t))\hat A_{t+1}$$

    下例第 1 条 transition 时间截断，第 2 条是 reset 后的新 episode。优势不能跨边界。

In [2]:
from ppo_pendulum.reference.algorithm import compute_gae
rewards = torch.tensor([1., 1., 2.])
values = torch.tensor([0.5, 0.25, 0.3])
next_values = torch.tensor([0.25, 4., 8.])
terminated = torch.tensor([False, False, True])
truncated = torch.tensor([False, True, False])
gamma, gae_lambda = 0.9, 0.95
bootstrap_mask = (~terminated).float()
recursion_mask = (~(terminated | truncated)).float()
deltas = rewards + gamma * bootstrap_mask * next_values - values
advantages, targets = compute_gae(rewards, values, next_values,
                                  terminated, truncated, gamma, gae_lambda)
for t in range(len(rewards)):
    print(f't={t}: bootstrap={bootstrap_mask[t]:.0f}, recurse={recursion_mask[t]:.0f}, '
          f'delta={deltas[t]:.4f}, advantage={advantages[t]:.4f}, target={targets[t]:.4f}')
torch.testing.assert_close(advantages[1], torch.tensor(4.35))
torch.testing.assert_close(targets[1], torch.tensor(4.6))
torch.testing.assert_close(targets[2], torch.tensor(2.0))
assert not advantages.requires_grad and not targets.requires_grad

t=0: bootstrap=1, recurse=1, delta=0.7250, advantage=4.4443, target=4.9443
t=1: bootstrap=1, recurse=0, delta=4.3500, advantage=4.3500, target=4.6000
t=2: bootstrap=0, recurse=0, delta=1.7000, advantage=1.7000, target=2.0000


## 2. 验证截断处不会泄漏新回合优势

大幅修改 reset 后的 reward，第 0、1 条的 advantage 应完全不变。

In [3]:
changed_rewards = rewards.clone()
changed_rewards[2] = 1000.
changed_advantages, _ = compute_gae(changed_rewards, values, next_values,
                                   terminated, truncated, gamma, gae_lambda)
torch.testing.assert_close(advantages[:2], changed_advantages[:2])
print('通过：reset 后奖励不影响前一回合优势。')

通过：reset 后奖励不影响前一回合优势。


## 3. 手写 TODO 与检查

编辑 `src/ppo_pendulum/exercises/gae.py`。补完后启用检查；同时建议用原项目的参考边界测试自己扩展测试用例。

In [4]:
RUN_EXERCISE = False
if RUN_EXERCISE:
    from ppo_pendulum.exercises.gae import compute_gae as student_gae
    student_adv, student_targets = student_gae(rewards, values, next_values,
                                              terminated, truncated, gamma, gae_lambda)
    torch.testing.assert_close(student_adv, advantages)
    torch.testing.assert_close(student_targets, targets)
    print('练习结果与已手算的参考结果一致。')
else:
    print('完成 gae.py 后启用练习检查。')

完成 gae.py 后启用练习检查。


## 下一步

打开 `04_ppo_clipped_loss.ipynb`，观察优势如何影响策略更新方向。

公式对照：[PPO_FORMULAS.md](../docs/PPO_FORMULAS.md)。